In [ ]:
from pathlib import Path
import sys
import torch

# Gestion du root projet
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import de tes outils (si tu décides de les placer dans ton package plus tard)
from pinkcc_ct_seg.utils import get_device

PROJECT_ROOT = /home/alouiyaz/projects/PINKCC/Brain_tumor_MRI_classification
SRC exists = True


In [3]:
# import
import matplotlib.pyplot as plt
from torch.utils.data import Subset, DataLoader

from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.transforms import get_eval_transforms
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.models.resnet18 import build_resnet18
from pinkcc_ct_seg.training.engine import predict_probabilities
from pinkcc_ct_seg.utils import get_device, load_checkpoint, set_seed
from pinkcc_ct_seg.evaluation.thresholding import (
    apply_threshold,
    scan_thresholds,
    select_threshold_by_best_f1,
    select_threshold_by_recall_constraint,
)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# config 
set_seed(42)
device = get_device()

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

DATA_DIR = PROJECT_ROOT / "data/raw/brain_mri/Training"
TEST_DIR = PROJECT_ROOT / "data/raw/brain_mri/Testing"
MODEL_PATH = PROJECT_ROOT / "models/best_resnet18_methodB_advanced.pt"

print("device =", device)
print("MODEL_PATH exists =", MODEL_PATH.exists())

In [ ]:
# Dataset/loaders
eval_tfms = get_eval_transforms(img_size=IMG_SIZE)

full_train_ds = BrainMRIDataset(DATA_DIR, transform=eval_tfms)
labels = [full_train_ds.samples[i][1] for i in range(len(full_train_ds))]

train_idx, val_idx = make_train_val_split(labels, val_size=0.2, random_state=42)

val_ds = Subset(full_train_ds, val_idx)
test_ds = BrainMRIDataset(TEST_DIR, transform=eval_tfms)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("Val size:", len(val_ds))
print("Test size:", len(test_ds))

In [ ]:
# charger le modèle
model = build_resnet18(num_classes=2, pretrained=False)
model = load_checkpoint(model, MODEL_PATH, map_location=device)
model = model.to(device)
model.eval()

print("Model loaded.")

In [ ]:
# probabilités sur le set de validation
y_val, probs_val = predict_probabilities(model, val_loader, device)

print("Nb val targets:", len(y_val))
print("Nb val probs:", len(probs_val))
print("Exemple probs:", probs_val[:5])

In [ ]:
# probabilités sur le set de validation
y_val, probs_val = predict_probabilities(model, val_loader, device)

print("Nb val targets:", len(y_val))
print("Nb val probs:", len(probs_val))
print("Exemple probs:", probs_val[:5])


In [ ]:
# Balayage des seuils
results = scan_thresholds(y_val, probs_val)
results[:5]

In [ ]:
# meilleur seuil F1
best_f1_threshold = select_threshold_by_best_f1(results)
best_f1_threshold

In [ ]:
# seuile avec recal sup 90%
best_recall90_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=None
)

best_recall90_threshold

In [ ]:
# seuil avec recall >=90% et precision >= 80%
best_clinical_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=0.80
)

best_clinical_threshold

In [ ]:
# courbe seuil /métriques
best_clinical_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=0.80
)

best_clinical_threshold

In [ ]:
# choisir un seuil finale
# variante F1 
final_threshold = best_f1_threshold["threshold"]
print("Seuil final (F1):", final_threshold)

# Variante métier 
final_threshold = best_clinical_threshold["threshold"]
print("Seuil final (métier):", final_threshold)


In [ ]:
# application au test finale
from sklearn.metrics import classification_report, confusion_matrix

y_test, probs_test = predict_probabilities(model, test_loader, device)
y_pred_test = apply_threshold(probs_test, final_threshold)

print("Threshold retenu:", final_threshold)
print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_test))

## Workflow 
1. entraînement du modèle
2. sauvegarde du meilleur modèle
3. calibration du seuil sur validation
4. choix du seuil selon une règle claire
5. test final avec ce seuil